In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import itertools
import time
import random
from tqdm import tqdm

# Sklearn & XGBoost
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from skimage.feature import hog


In [2]:

# Cấu hình hiển thị
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# ===================================================================
# 0. SETUP
# ===================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
SEED = 42
seed_everything(SEED)

DATA_SOURCES = {
    'fogsmog': '/kaggle/input/weather-dataset/dataset/fogsmog',
    'rain': '/kaggle/input/weather-dataset/dataset/rain',
    'shine': '/kaggle/input/multiclass-weather-dataset/Multi-class Weather Dataset/Shine'
}
IMG_SIZE = 256

print("[0] LOADING PATHS & CACHING GRAYSCALE IMAGES...")
img_paths, labels_raw = [], []
for label, path in DATA_SOURCES.items():
    if os.path.exists(path):
        files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        img_paths.extend(files)
        labels_raw.extend([label] * len(files))

le = LabelEncoder()
y_encoded = le.fit_transform(labels_raw)

# --- CHỈ CẦN CACHE ẢNH XÁM (Đỡ phải đọc lại ảnh) ---
gray_cache = []
for path in tqdm(img_paths):
    img = cv2.imread(path)
    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_cache.append(gray)

print(f"Loaded {len(gray_cache)} images for HOG tuning.")

# ===================================================================
# 1. DEFINE GRID SEARCH SPACE
# ===================================================================
# Không gian tìm kiếm
hog_params = {
    'pixels_per_cell': [4, 8, 16, 32], 
    'orientations': [6, 9, 12],
    'cells_per_block': [2, 3]
}

# Tạo danh sách tổ hợp (3 x 3 x 2 = 18 tổ hợp)
hog_combinations = list(itertools.product(
    hog_params['pixels_per_cell'],
    hog_params['orientations'],
    hog_params['cells_per_block']
))

print(f"\n[1] Starting Grid Search on {len(hog_combinations)} configurations...")

# ===================================================================
# 2. RUN SEARCH (HOG ONLY)
# ===================================================================
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

start_time = time.time()

for ppc, orient, cpb in hog_combinations:
    config_name = f"PPC={ppc}, Ori={orient}, CPB={cpb}"
    
    # 1. Tính HOG cho toàn bộ dataset với tham số này
    hog_features = []
    for gray in gray_cache:
        fd = hog(gray, orientations=orient, 
                 pixels_per_cell=(ppc, ppc), 
                 cells_per_block=(cpb, cpb), 
                 visualize=False, feature_vector=True)
        # Rút gọn thống kê (Mean, Std, Max) - Giống logic các bài trước
        hog_features.append([np.mean(fd), np.std(fd), np.max(fd)])
    
    X = np.array(hog_features)
    
    # 2. Đánh giá bằng XGBoost Default
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', xgb)])
    scores = cross_val_score(pipe, X, y_encoded, cv=cv, scoring='f1_macro', n_jobs=-1)
    mean_f1 = scores.mean()
    
    print(f"   [{config_name}] -> F1: {mean_f1:.4f}")
    
    results.append({
        'PPC': ppc,
        'Orientations': orient,
        'Block': cpb,
        'F1-Macro': mean_f1
    })

print("\n" + "="*60)
print(f" DONE in {(time.time() - start_time)/60:.1f} minutes")
print("="*60)

# ===================================================================
# 3. SHOW BEST PARAMS
# ===================================================================
df_res = pd.DataFrame(results)
df_res = df_res.sort_values(by='F1-Macro', ascending=False)

print("\n>>> BEST HOG CONFIGURATION:")
print(df_res.head(5).to_string(index=False))

# Lấy dòng tốt nhất ra
best_row = df_res.iloc[0]
print(f"\n✅ RECOMMENDED SETTINGS FOR FINAL CODE:")
print(f"   pixels_per_cell = ({int(best_row['PPC'])}, {int(best_row['PPC'])})")
print(f"   orientations = {int(best_row['Orientations'])}")
print(f"   cells_per_block = ({int(best_row['Block'])}, {int(best_row['Block'])})")

[0] LOADING PATHS & CACHING GRAYSCALE IMAGES...


100%|██████████| 1630/1630 [00:14<00:00, 108.80it/s]


Loaded 1630 images for HOG tuning.

[1] Starting Grid Search on 24 configurations...
   [PPC=4, Ori=6, CPB=2] -> F1: 0.5179
   [PPC=4, Ori=6, CPB=3] -> F1: 0.4925
   [PPC=4, Ori=9, CPB=2] -> F1: 0.4968
   [PPC=4, Ori=9, CPB=3] -> F1: 0.4874
   [PPC=4, Ori=12, CPB=2] -> F1: 0.5525
   [PPC=4, Ori=12, CPB=3] -> F1: 0.5580
   [PPC=8, Ori=6, CPB=2] -> F1: 0.5292
   [PPC=8, Ori=6, CPB=3] -> F1: 0.4827
   [PPC=8, Ori=9, CPB=2] -> F1: 0.5021
   [PPC=8, Ori=9, CPB=3] -> F1: 0.4944
   [PPC=8, Ori=12, CPB=2] -> F1: 0.5535
   [PPC=8, Ori=12, CPB=3] -> F1: 0.5291
   [PPC=16, Ori=6, CPB=2] -> F1: 0.5397
   [PPC=16, Ori=6, CPB=3] -> F1: 0.5055
   [PPC=16, Ori=9, CPB=2] -> F1: 0.5203
   [PPC=16, Ori=9, CPB=3] -> F1: 0.4953
   [PPC=16, Ori=12, CPB=2] -> F1: 0.5945
   [PPC=16, Ori=12, CPB=3] -> F1: 0.5216
   [PPC=32, Ori=6, CPB=2] -> F1: 0.5055
   [PPC=32, Ori=6, CPB=3] -> F1: 0.5180
   [PPC=32, Ori=9, CPB=2] -> F1: 0.5179
   [PPC=32, Ori=9, CPB=3] -> F1: 0.4892
   [PPC=32, Ori=12, CPB=2] -> F1: 0.5899
